<a href="https://colab.research.google.com/github/sarthak-geek/Book-Recommendation-Engine-using-KNN/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [413]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [414]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-06-29 07:08:00--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.1’

book-crossings.zip. 100%[===================>]  24.88M   120MB/s    in 0.2s    

2026-06-29 07:08:01 (120 MB/s) - ‘book-crossings.zip.1’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace BX-Books.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace BX-Users.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [436]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [452]:
df_books[df_books['title'] == "Where the Heart Is (Oprah's Book Club (Paperback))"]

,isbn,title,author
706,0446672211,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts


In [448]:
uniq_users = df_ratings['user'].value_counts()
uniq_users = uniq_users[uniq_users > 200]
most_rated = df_ratings['isbn'].value_counts()
most_rated = most_rated[most_rated > 100]
df_ratings = df_ratings[df_ratings['user'].isin(uniq_users.index)]

In [465]:
readed_by = df_ratings[df_ratings['isbn'] == '0446672211']['user']
new_ds = df_ratings[df_ratings['user'].isin(readed_by)]

In [466]:
pivoted = new_ds.pivot_table(index='isbn', columns='user', values='rating').fillna(0)

In [467]:
model = NearestNeighbors(metric='cosine')
model.fit(pivoted.values)

NearestNeighbors(metric='cosine')

In [478]:
dist, ind = model.kneighbors([pivoted.loc['0446672211'].values])
# pivoted.loc[ind]
print(dist[0])
print(df_books[df_books['isbn'].isin(pivoted.iloc[ind[0]].index)])
# print(pivoted.iloc[ind[0]])



[0.         0.55242735 0.6701081  0.6892542  0.7017829 ]
            isbn                                              title  \
45    0671888587                                 I'll Be Seeing You   
252   0316782505                                The Weight of Water   
408   0316666343                          The Lovely Bones: A Novel   
706   0446672211  Where the Heart Is (Oprah's Book Club (Paperba...   
3703  0345447840                                        The Surgeon   

                  author  
45    Mary Higgins Clark  
252         Anita Shreve  
408         Alice Sebold  
706         Billie Letts  
3703      TESS GERRITSEN  


Index(['0446672211', '0316666343', '0671888587', '0345447840', '0316782505'], dtype='object', name='isbn')

InvalidIndexError: (0         False
1         False
2         False
3         False
4         False
          ...  
271374    False
271375    False
271376    False
271377    False
271378    False
Name: isbn, Length: 271379, dtype: bool, np.float32(0.0))

In [ ]:
# add your code here - consider creating a new cell for each section of code

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):

  recommended_books = pd.DataFrame({
      'title' : df_books.iloc[indice[0]].index.values, 'distance': distance[0]}).sort_values(by='distance', ascending=False).head(5).values
  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()